# Multi-turn chat, streamed

`stream_generate` yields chunks as they are sampled, so a reply can be printed
while it is still being produced.

The one thing to get right is prompt construction: re-template the **whole**
message list every turn, assistant turns included. Sending only the newest turn
loses the conversation.

In [1]:
from mlx_vlm import apply_chat_template, load, stream_generate

MODEL = "mlx-community/Qwen3-VL-8B-Instruct-4bit"
IMAGE = "../images/cats.jpg"

model, processor = load(MODEL)
messages = []

In [2]:
def ask(question, max_tokens=80):
    messages.append({"role": "user", "content": question})
    prompt = apply_chat_template(processor, model.config, messages, num_images=1)

    print(f"You: {question}\nAssistant: ", end="")
    reply = ""
    for chunk in stream_generate(
        model, processor, prompt, image=[IMAGE], max_tokens=max_tokens, temperature=0.0
    ):
        print(chunk.text, end="")
        reply += chunk.text

    messages.append({"role": "assistant", "content": reply})
    print()

In [3]:
ask("What animals are in this image?")

You: What animals are in this image?
Assistant: 

Based on the image provided, the animals are **cats**.

There are two tabby cats sleeping on a

 bright pink couch or cushion. One cat is lying on its side, while the other is lying on its stomach with

 its head down. Both appear to be relaxed and asleep.


In [4]:
ask("What colour is the couch they are on?")

You: What colour is the couch they are on?
Assistant: 

The couch (or cushion) the cats are sleeping on is **bright pink**.

It has a

 soft, textured fabric, and the two tabby cats are lying on it alongside two remote controls.


The second answer resolves "they" from the first turn, which is the whole point
of resending the message list. Note the image is re-encoded on every turn —
nothing is cached unless the model implements the vision-cache hook, and most
do not.